# 3D Printable Mount For Val1 BotSat

This file is a Jupyter notebook that contains the code for the 3D printable mount for the Val1 BotSat. 

It uses the `cadquery` and `jupyter_cadquery` library to create the 3D model for printing.

We also use the `cq_warehouse` library to get the standard parts for the mount, such as threaded holes for fastening the mount to the Val1 BotSat bottle.

In [ ]:
import cadquery as cq
import jupyter_cadquery as jcq

from cq_warehouse.fastener import SocketHeadCapScrew
import cq_warehouse.fastener as fst
from cq_warehouse.bearing import SingleRowDeepGrooveBallBearing
import cq_warehouse.extensions

# Remove these lines if you prefer rendering the different steps inline instead. It means you can see how the different steps differ, 
# but it also makes it more difficult to see the code because it's all spread out. Your choice!
jcq.set_defaults(axes=True, timeit=False)
jcq.open_viewer("Jupyter CadQuery - Example", cad_width=640, height=480, glass=True)

We allow for basic configuration of the mount. And good thing we did as well as the first version we made was too big for the 
Fanta Exotic 1.5L bottle we used. 

You can adjust these to fit your own bottle and perfboard sizes.

In [ ]:
bottle_dia = 89

wall_height = 20
wall_thickness = 5

perfboard_dim = (46, 66)
perfboard_padding = 7

central_hole_dia = bottle_dia / 1.5

We're also going to use M4 screws to secure the mount inside the bottle. We're also creating M4 holes here for attaching the perfboard, although our board unfortunately had holes that were too small, so we just secured the perfboard to the mount with zip ties. You could create a different screw size for the perfboard if you want by declaring a `screw_board` variable and using that for the holes in the bottom of the mount.

In [ ]:
screw = fst.ButtonHeadScrew(
    size="M4-0.7", length=10, fastener_type="iso7380_1", simple=False
)
screw

We're now ready to create the basic shape of the mount. It's a cylinder that fits inside the bottle and has a flat bottom 
that we can attach the perfboard to. It also has a hole in the centre to save weight.

In [ ]:
botsat_mount = cq.Assembly(None, name="botsat_mount")

mount = (
    cq.Workplane("XY")
    .circle(bottle_dia/2)
    .extrude(wall_height)
    .faces(">Z")
    .workplane()
    .tag("top")
    .circle((bottle_dia-wall_thickness*2)/2)
    .cutBlind(-wall_height+wall_thickness)
    .workplaneFromTagged("top")
    .rect(perfboard_dim[0]+perfboard_padding, perfboard_dim[1]+perfboard_padding)
    .cutBlind(-wall_height+wall_thickness)
    .workplaneFromTagged("top")
    .circle(central_hole_dia/2)
    .cutBlind(-wall_height)
    .faces("<Z")
    .workplane()
    .rect(*perfboard_dim, forConstruction=True)
    .vertices()
    .threadedHole(fastener=screw, depth=5, counterSunk=False, baseAssembly=botsat_mount)
)
mount

Now to create the threaded holes around the circumference of the bottle we hit a small snag. Turns out the geometries
created when putting a threaded hole into a curved surface must be pretty complex, and they took an unbearably long time to create. 

So, to simplify this shape, we put a cuboid running through the centre of the cylinder and add the threaded holes to this. This also has the added benefit of creating a small flat patch on the outside of the cylinder to spread the load when the screw and washer is screwed in from the outside. 

In [ ]:
mount = (
    cq.Workplane("XZ")
    .tag("base")
    .add(mount)
    .center(0,10)
    .workplane(offset=bottle_dia/2)
    .tag("side")
    .rect(10, 10)
    .extrude(-bottle_dia)
    .workplaneFromTagged("side")
    .threadedHole(fastener=screw, depth=5, counterSunk=False, baseAssembly=botsat_mount)
    .faces(">Y")
    .workplane()
    .threadedHole(fastener=screw, depth=5, counterSunk=False, baseAssembly=botsat_mount)
)
mount

That looks good, so let's make one in the other dimension as well.

In [ ]:
mount = (
    cq.Workplane("YZ")
    .tag("base")
    .add(mount)
    .center(0,10)
    .workplane(offset=bottle_dia/2)
    .tag("side")
    .rect(10, 10)
    .extrude(-bottle_dia)
    .workplaneFromTagged("side")
    .threadedHole(fastener=screw, depth=5, counterSunk=False, baseAssembly=botsat_mount)
    .faces("<X")
    .workplane()
    .threadedHole(fastener=screw, depth=5, counterSunk=False, baseAssembly=botsat_mount)
)
mount

Finally we need to remove the central bits of these supports, so let's snip out the hole in the cylinder again.

In [ ]:
mount = (
    cq.Workplane("XY")
    .add(mount)
    .faces(">Z")
    .workplane()
    .circle((bottle_dia-wall_thickness*2)/2)
    .cutBlind(-wall_height+wall_thickness)
)
mount

In [ ]:
botsat_mount.add(mount, name="mount", color=cq.Color(162 / 255, 138 / 255, 255 / 255))

botsat_mount

In [ ]:
botsat_mount.fastenerQuantities()

In [ ]:
cq.exporters.export(mount, "botsat-mount.stl")